# 02 Peak QC
This notebook details processes to QC peaks for RUNX1 and RUNX3 from day 5 shCd19 and shRunx3.

## 01.01 Initialize Environment
Load packages needed and move to working directory.

In [1]:
# Load Libraries
library(dplyr)
library(tidyr)

# Load data
work_dir     <- "/home/dalbao/AlbaoRunx3Manuscript/cutnrun"

# Add peak type and date to prefix
# prefix <- paste0(format(Sys.Date(), "%y%m%d"), "-")
prefix <- ""

# Move to working directory
setwd(work_dir)


Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




## 01.02 Define Peak QC Function
Adapt `source_data/analyze_macs2_narrowPeak.R` into a reusable function. For each MACS2 `narrowPeak` file it renders four QC plots (signalValue histogram, qValue histogram, signalValue vs qValue, and a signalValue scree plot with the inflection/"knee" point marked) as PDFs (not attached to the notebook), and writes a qValue cutoff table and a BED file of peaks above the inflection point. All outputs are written into `02_peakqc/`.

In [2]:
# Peak QC constants
peak_dir   <- "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2"
output_dir <- "02_peakqc"
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

peak_cols <- c("chrom", "start", "end", "name", "score", "strand",
               "signalValue", "pValue", "qValue", "summit")
peak_classes <- c("character", "integer", "integer", "character", "numeric", "character",
                   "numeric", "numeric", "numeric", "integer")

col_main   <- "#2E6FA3"
col_grid   <- "grey88"
col_accent <- "#C0392B"
col_muted  <- "grey75"

# ---- Plot-drawing helpers (PDF export only, not attached to the notebook) ----

draw_signalValue_hist <- function(h, base_name) {
  plot(NA, xlim = range(h$breaks), ylim = c(0, max(h$counts) * 1.08),
       xlab = "signalValue", ylab = "Number of peaks",
       main = sprintf("Distribution of signalValue: %s", base_name), bty = "l")
  grid(col = col_grid, lty = 1)
  plot(h, col = col_main, border = "white", add = TRUE)
}

draw_qvalue_hist <- function(h, base_name) {
  plot(NA, xlim = range(h$breaks), ylim = c(0, max(h$counts) * 1.08),
       xlab = expression(-log[10](qvalue)), ylab = "Number of peaks",
       main = sprintf("Distribution of qValue: %s", base_name), bty = "l")
  grid(col = col_grid, lty = 1)
  plot(h, col = col_main, border = "white", add = TRUE)
}

draw_signal_vs_qvalue <- function(neglog10_q, signal, base_name) {
  plot(neglog10_q, signal, type = "n",
       xlab = expression(-log[10](qvalue)), ylab = "signalValue",
       main = sprintf("signalValue vs qValue: %s", base_name), bty = "l")
  grid(col = col_grid, lty = 1)
  points(neglog10_q, signal, pch = 16, cex = 0.6,
         col = adjustcolor(col_main, alpha.f = 0.45))
}

draw_signalValue_scree <- function(rank_x, sorted_signal, point_col,
                                    knee_rank, knee_signal, knee_qvalue, base_name) {
  plot(rank_x, sorted_signal, type = "n",
       xlab = "Rank", ylab = "signalValue",
       main = sprintf("Scree plot of signalValue: %s", base_name), bty = "l")
  grid(col = col_grid, lty = 1)
  lines(rank_x, sorted_signal, col = col_muted, lwd = 1)
  points(rank_x, sorted_signal, pch = 16, cex = 0.5,
         col = adjustcolor(point_col, alpha.f = 0.6))
  label_x <- max(rank_x) * 0.32
  label_y <- max(sorted_signal) * 0.55
  arrows(label_x, label_y,
         knee_rank + diff(range(rank_x)) * 0.01, knee_signal + diff(range(sorted_signal)) * 0.02,
         length = 0.08, col = "black", lwd = 1)
  points(knee_rank, knee_signal, pch = 21, cex = 1.3, bg = "white", col = "black", lwd = 1.5)
  text(label_x, label_y,
       labels = sprintf("Rank = %d\nsignalValue = %.2f\nqValue = %.2f",
                         knee_rank, knee_signal, knee_qvalue),
       col = "black", pos = 4, cex = 0.8)
  legend("topright", legend = c("qValue > inflection", "qValue <= inflection"),
         col = c(col_accent, col_main), pch = 16, bty = "n", cex = 0.8)
}

# Draw a plot to a PDF device only; never attached as notebook cell output.
render_plot <- function(pdf_file, draw_fn, width = 6, height = 5) {
  pdf(pdf_file, width = width, height = height)
  draw_fn()
  dev.off()
}

# ---- Main QC routine, adapted from source_data/analyze_macs2_narrowPeak.R ----

analyze_narrowpeak <- function(input_file, output_dir) {
  base_name  <- sub("\\.narrowPeak$", "", basename(input_file))
  out_prefix <- file.path(output_dir, paste0(prefix, base_name))

  cat(sprintf("== %s ==\n", base_name))

  # Drop UCSC track/comment lines and blank lines some pipelines prepend.
  raw_lines <- readLines(input_file)
  raw_lines <- raw_lines[nzchar(raw_lines) & !grepl("^(track|#)", raw_lines)]
  if (length(raw_lines) == 0) {
    message(sprintf(
      "No peaks found in %s (0 peaks called) - skipping plots, writing an empty cutoff table and BED file.",
      input_file))
    empty_table <- data.frame(qvalue_cutoff = character(0),
                               neg_log10_qvalue_threshold = integer(0),
                               n_peaks_passing = integer(0))
    write.table(empty_table, sprintf("%s.qvalue_cutoff_table.tsv", out_prefix),
                sep = "\t", row.names = FALSE, quote = FALSE)
    file.create(sprintf("%s.above_inflection.bed", out_prefix))
    return(invisible(list(base_name = base_name, n_peaks = 0L, n_above_inflection = 0L,
                           knee_rank = NA, knee_signal = NA, knee_qvalue = NA)))
  }

  peaks <- read.table(
    text = raw_lines, sep = "\t", stringsAsFactors = FALSE,
    col.names = peak_cols, colClasses = peak_classes
  )

  bad_rows <- which(is.na(peaks$signalValue) | is.na(peaks$qValue))
  if (length(bad_rows) > 0) {
    stop(sprintf(
      "%d row(s) have a missing/non-numeric signalValue or qValue (e.g. line %d: %s)",
      length(bad_rows), bad_rows[1], raw_lines[bad_rows[1]]
    ), call. = FALSE)
  }

  signal <- peaks$signalValue
  # MACS2 narrowPeak stores qValue as -log10(qvalue); larger = more significant.
  neglog10_q <- peaks$qValue

  # ---- 1. Histogram of signalValue -----------------------------------------
  h <- hist(signal, breaks = 50, plot = FALSE)
  render_plot(sprintf("%s.signalValue_hist.pdf", out_prefix),
              function() draw_signalValue_hist(h, base_name))

  # ---- 2. Histogram of qValue -----------------------------------------------
  h <- hist(neglog10_q, breaks = 50, plot = FALSE)
  render_plot(sprintf("%s.qvalue_hist.pdf", out_prefix),
              function() draw_qvalue_hist(h, base_name))

  # ---- 3. signalValue (y) vs qValue (x) -------------------------------------
  render_plot(sprintf("%s.signal_vs_qvalue.pdf", out_prefix),
              function() draw_signal_vs_qvalue(neglog10_q, signal, base_name))

  # ---- 4. Scree plot: signalValue (y) vs rank (x) ---------------------------
  ord <- order(signal, decreasing = TRUE)
  sorted_signal <- signal[ord]
  sorted_qval <- neglog10_q[ord]
  rank_x <- seq_along(sorted_signal)

  # Inflection ("knee") point: the point on the curve with the greatest
  # perpendicular distance from the chord connecting its two endpoints.
  x_norm <- (rank_x - min(rank_x)) / (max(rank_x) - min(rank_x))
  y_norm <- (sorted_signal - min(sorted_signal)) / (max(sorted_signal) - min(sorted_signal))
  x1 <- x_norm[1]; y1 <- y_norm[1]
  x2 <- x_norm[length(x_norm)]; y2 <- y_norm[length(y_norm)]
  chord_dist <- abs((y2 - y1) * x_norm - (x2 - x1) * y_norm + x2 * y1 - y2 * x1) /
    sqrt((y2 - y1)^2 + (x2 - x1)^2)
  knee_idx <- which.max(chord_dist)
  knee_rank <- rank_x[knee_idx]
  knee_signal <- sorted_signal[knee_idx]
  knee_qvalue <- sorted_qval[knee_idx]

  # Peaks more significant than the inflection point (qValue is -log10(qvalue),
  # so "higher" qValue means a more significant, smaller actual q-value).
  above_knee <- sorted_qval > knee_qvalue
  point_col <- ifelse(above_knee, col_accent, col_main)

  render_plot(sprintf("%s.signalValue_scree.pdf", out_prefix),
              function() draw_signalValue_scree(rank_x, sorted_signal, point_col,
                                                 knee_rank, knee_signal, knee_qvalue, base_name))

  # ---- 5. Peak counts at successively stricter qValue cutoffs -------------
  # cutoff of 1e-N corresponds to -log10(qvalue) >= N
  max_order <- floor(max(neglog10_q))
  orders <- seq_len(max_order)
  n_peaks <- vapply(orders, function(o) sum(neglog10_q >= o), integer(1))

  cutoff_table <- data.frame(
    qvalue_cutoff = sprintf("1e-%d", orders),
    neg_log10_qvalue_threshold = orders,
    n_peaks_passing = n_peaks
  )
  write.table(cutoff_table, sprintf("%s.qvalue_cutoff_table.tsv", out_prefix),
              sep = "\t", row.names = FALSE, quote = FALSE)

  # ---- 6. BED file of peaks above the scree plot inflection point ---------
  peaks_above <- peaks[signal > knee_signal, ]
  peaks_above <- peaks_above[order(peaks_above$chrom, peaks_above$start), ]
  write.table(peaks_above, sprintf("%s.above_inflection.bed", out_prefix),
              sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)

  cat(sprintf(
    "Scree plot inflection point: rank = %d, signalValue = %.4f, qValue (-log10) = %.4f (raw qValue ~ %.3g)\n",
    knee_rank, knee_signal, knee_qvalue, 10^(-knee_qvalue)))
  cat(sprintf("%d peaks with signalValue above the inflection point written to %s.above_inflection.bed\n",
              nrow(peaks_above), out_prefix))
  cat(sprintf("Done. %d peaks read from %s.\n\n", nrow(peaks), input_file))

  invisible(list(base_name = base_name, n_peaks = nrow(peaks), n_above_inflection = nrow(peaks_above),
                 knee_rank = knee_rank, knee_signal = knee_signal, knee_qvalue = knee_qvalue))
}

## 01.03 Run Peak QC on Runx narrowPeak Files
Find the merged-replicate MACS2 `narrowPeak` files for RUNX1/RUNX3 (`*_Runx*narrowPeak`) and run the QC function on each.

In [3]:
# Peaks matching *_Runx*narrowPeak
narrowpeak_files <- list.files(peak_dir, pattern = "_Runx.*narrowPeak$", full.names = TRUE)
narrowpeak_files

# Run QC on each peak file; outputs (plots + PDFs + tables) land in output_dir
qc_results <- lapply(narrowpeak_files, analyze_narrowpeak, output_dir = output_dir)
names(qc_results) <- basename(narrowpeak_files)

[1] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/early_Runx1.merged.macs2_peaks.narrowPeak"   
 [2] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/early_Runx3.merged.macs2_peaks.narrowPeak"   
 [3] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/late_Runx1.merged.macs2_peaks.narrowPeak"    
 [4] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/late_Runx3.merged.macs2_peaks.narrowPeak"    
 [5] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/memory_Runx1.merged.macs2_peaks.narrowPeak"  
 [6] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/memory_Runx3.merged.macs2_peaks.narrowPeak"  
 [7] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/shCd19_Runx1.merged.macs2_peaks.narrowPeak"  
 [8] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/shCd19_Runx3.merged.macs2_peaks.narrowPeak"  
 [9] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/shRunx3_Runx1.merged.macs2_peaks.narrowPeak" 
[10] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/shRunx3_Runx3.merged.macs2_peaks.narrowPeak" 
[11] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/terminal_Runx1.merged.macs2_peaks.narrowPeak"
[12] "source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/terminal_Runx3.merged.macs2_peaks.narrowPeak"

== early_Runx1.merged.macs2_peaks ==


Scree plot inflection point: rank = 1003, signalValue = 6.0928, qValue (-log10) = 4.6082 (raw qValue ~ 2.46e-05)
1002 peaks with signalValue above the inflection point written to 02_peakqc/early_Runx1.merged.macs2_peaks.above_inflection.bed
Done. 9177 peaks read from source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/early_Runx1.merged.macs2_peaks.narrowPeak.

== early_Runx3.merged.macs2_peaks ==


Scree plot inflection point: rank = 620, signalValue = 6.4340, qValue (-log10) = 5.1945 (raw qValue ~ 6.39e-06)
619 peaks with signalValue above the inflection point written to 02_peakqc/early_Runx3.merged.macs2_peaks.above_inflection.bed
Done. 7697 peaks read from source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/early_Runx3.merged.macs2_peaks.narrowPeak.

== late_Runx1.merged.macs2_peaks ==


Scree plot inflection point: rank = 1179, signalValue = 6.3876, qValue (-log10) = 5.3285 (raw qValue ~ 4.69e-06)
1178 peaks with signalValue above the inflection point written to 02_peakqc/late_Runx1.merged.macs2_peaks.above_inflection.bed
Done. 11688 peaks read from source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/late_Runx1.merged.macs2_peaks.narrowPeak.

== late_Runx3.merged.macs2_peaks ==


...

Scree plot inflection point: rank = 2888, signalValue = 11.2501, qValue (-log10) = 16.3680 (raw qValue ~ 4.29e-17)
2887 peaks with signalValue above the inflection point written to 02_peakqc/shCd19_Runx3.merged.macs2_peaks.above_inflection.bed
Done. 131978 peaks read from source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/shCd19_Runx3.merged.macs2_peaks.narrowPeak.

== shRunx3_Runx1.merged.macs2_peaks ==


Scree plot inflection point: rank = 3577, signalValue = 8.0968, qValue (-log10) = 8.9635 (raw qValue ~ 1.09e-09)
3576 peaks with signalValue above the inflection point written to 02_peakqc/shRunx3_Runx1.merged.macs2_peaks.above_inflection.bed
Done. 94102 peaks read from source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/shRunx3_Runx1.merged.macs2_peaks.narrowPeak.

== shRunx3_Runx3.merged.macs2_peaks ==


Scree plot inflection point: rank = 2853, signalValue = 8.9848, qValue (-log10) = 10.7382 (raw qValue ~ 1.83e-11)
2852 peaks with signalValue above the inflection point written to 02_peakqc/shRunx3_Runx3.merged.macs2_peaks.above_inflection.bed
Done. 109974 peaks read from source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/shRunx3_Runx3.merged.macs2_peaks.narrowPeak.

== terminal_Runx1.merged.macs2_peaks ==


Scree plot inflection point: rank = 3244, signalValue = 6.5491, qValue (-log10) = 6.3551 (raw qValue ~ 4.41e-07)
3243 peaks with signalValue above the inflection point written to 02_peakqc/terminal_Runx1.merged.macs2_peaks.above_inflection.bed
Done. 57451 peaks read from source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/terminal_Runx1.merged.macs2_peaks.narrowPeak.

== terminal_Runx3.merged.macs2_peaks ==


Scree plot inflection point: rank = 1207, signalValue = 7.7630, qValue (-log10) = 8.0001 (raw qValue ~ 1e-08)
1206 peaks with signalValue above the inflection point written to 02_peakqc/terminal_Runx3.merged.macs2_peaks.above_inflection.bed
Done. 39934 peaks read from source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/07_merged_replicates/03_called_peaks/macs2/terminal_Runx3.merged.macs2_peaks.narrowPeak.



## 01.04 Summary of Peaks Retained
Number of peaks called and number/percent retained above the scree plot inflection point for each peak set.

In [4]:
# Summarize peaks retained (above the inflection point) for each peak set
summary_df <- dplyr::bind_rows(lapply(qc_results, function(res) {
  data.frame(peak_set           = res$base_name,
             n_peaks            = res$n_peaks,
             n_above_inflection = res$n_above_inflection,
             pct_retained       = ifelse(res$n_peaks > 0,
                                          100 * res$n_above_inflection / res$n_peaks, NA))
}))

write.table(summary_df, file.path(output_dir, paste0(prefix, "peak_retention_summary.tsv")),
            sep = "\t", row.names = FALSE, quote = FALSE)

summary_df

peak_set,n_peaks,n_above_inflection,pct_retained
<chr>,<int>,<int>,<dbl>
early_Runx1.merged.macs2_peaks,9177,1002,10.918601
early_Runx3.merged.macs2_peaks,7697,619,8.042094
late_Runx1.merged.macs2_peaks,11688,1178,10.078713
late_Runx3.merged.macs2_peaks,13124,921,7.017678
memory_Runx1.merged.macs2_peaks,14711,1728,11.746312
memory_Runx3.merged.macs2_peaks,16936,1218,7.191781
shCd19_Runx1.merged.macs2_peaks,83851,3305,3.941515
shCd19_Runx3.merged.macs2_peaks,131978,2887,2.187486
shRunx3_Runx1.merged.macs2_peaks,94102,3576,3.800132
